### Data Ingestion to Vector Database Pipeline

In [24]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [25]:
### Read all PDFs inside the direcroty

def process_pdfs(pdf_directory):
    all_docs = []
    pdf_dir = Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files in the directory.")
    
    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file.name}")
        
        try:
            # Load the PDF using PyPDFLoader
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # If no documents were loaded, try PyMuPDFLoader
            if not documents:
                print(f"No content found with PyPDFLoader for {pdf_file}. Trying PyMuPDFLoader...")
                loader = PyMuPDFLoader(str(pdf_file))
                documents = loader.load()
            
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
                
            all_docs.extend(documents)
            
            print(f"Loaded {len(documents)} pages from {pdf_file}.")
            
        except Exception as e:
            print(f" Error: {e}")
            
    print(f"Total documents loaded: {len(all_docs)}")
    return all_docs


all_pdf_documents = process_pdfs("../data")
                     
            

Found 2 PDF files in the directory.
Processing file: LangChain.pdf
Loaded 14 pages from ..\data\pdf\LangChain.pdf.
Processing file: Understanding RAG — AI Jargon Simplified #1.pdf
Loaded 18 pages from ..\data\pdf\Understanding RAG — AI Jargon Simplified #1.pdf.
Total documents loaded: 32


In [26]:
all_pdf_documents


[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-11-10T11:43:58+00:00', 'author': '', 'keywords': '', 'moddate': '2024-11-10T11:43:58+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\LangChain.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': 'LangChain.pdf', 'file_type': 'pdf'}, page_content='LangChain\nVasilios Mavroudis\nAlan Turing Institute\nvmavroudis@turing.ac.uk\nAbstract. LangChain is a rapidly emerging framework that offers a ver-\nsatile and modular approach to developing applications powered by large\nlanguage models (LLMs). By leveraging LangChain, developers can sim-\nplify complex stages of the application lifecycle—such as development,\nproductionization, and deployment—making it easier to build scalable,\nstateful, and contextually aware applications. I

In [27]:
### Chunking 
# chunking startegies

def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    if split_docs:
        print(f"\n Example chunk")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs



In [28]:
chunks = split_documents(all_pdf_documents)
chunks

Split 32 documents into 70 chunks

 Example chunk
Content: LangChain
Vasilios Mavroudis
Alan Turing Institute
vmavroudis@turing.ac.uk
Abstract. LangChain is a rapidly emerging framework that offers a ver-
satile and modular approach to developing applications...
Metadata: {'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-11-10T11:43:58+00:00', 'author': '', 'keywords': '', 'moddate': '2024-11-10T11:43:58+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\LangChain.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': 'LangChain.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-11-10T11:43:58+00:00', 'author': '', 'keywords': '', 'moddate': '2024-11-10T11:43:58+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\LangChain.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': 'LangChain.pdf', 'file_type': 'pdf'}, page_content='LangChain\nVasilios Mavroudis\nAlan Turing Institute\nvmavroudis@turing.ac.uk\nAbstract. LangChain is a rapidly emerging framework that offers a ver-\nsatile and modular approach to developing applications powered by large\nlanguage models (LLMs). By leveraging LangChain, developers can sim-\nplify complex stages of the application lifecycle—such as development,\nproductionization, and deployment—making it easier to build scalable,\nstateful, and contextually aware applications. I

### Embedding and vectorDB


In [29]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings 
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [30]:
class EmbeddingManager:
    def __init__(self, model_name = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        try:
            print(f"Loading embedded model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            
            print(f"Model loaded successfully. Embedding dimention: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading Model {self.model_name}: {e}")
            raise
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        
        
        print(f"generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    
embedding_manager = EmbeddingManager()
embedding_manager   

Loading embedded model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4594.22it/s]


Model loaded successfully. Embedding dimention: 384


In [31]:
# Vector db store
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
   
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
  
            documents_text.append(doc.page_content)
            
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 98


In [32]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-11-10T11:43:58+00:00', 'author': '', 'keywords': '', 'moddate': '2024-11-10T11:43:58+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\LangChain.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': 'LangChain.pdf', 'file_type': 'pdf'}, page_content='LangChain\nVasilios Mavroudis\nAlan Turing Institute\nvmavroudis@turing.ac.uk\nAbstract. LangChain is a rapidly emerging framework that offers a ver-\nsatile and modular approach to developing applications powered by large\nlanguage models (LLMs). By leveraging LangChain, developers can sim-\nplify complex stages of the application lifecycle—such as development,\nproductionization, and deployment—making it easier to build scalable,\nstateful, and contextually aware applications. I

In [33]:
# Chunks to embeddings
#texts
texts = [doc.page_content for doc in chunks]

#generate embeddings

embeddings = embedding_manager.generate_embeddings(texts)

#store into db
vectorstore.add_documents(chunks, embeddings)

generating embeddings for 70 texts...


Batches: 100%|██████████| 3/3 [00:06<00:00,  2.24s/it]


Generated embeddings with shape: (70, 384)
Adding 70 documents to vector store...
Successfully added 70 documents to vector store
Total documents in collection: 168


### RETRIEVER


In [34]:
class RAGRetriever:
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):

                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [35]:
rag_retriever

In [36]:
rag_retriever.retrieve("What is langchain?") #Retriver = context + prompt

Retrieving documents for query: 'What is langchain?'
Top K: 5, Score threshold: 0.0
generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 84.44it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_3ba6337e_45',
  'content': '14 Vasilios Mavroudis\nReferences\n1. Josh Achiam, Steven Adler, Sandhini Agarwal, Lama Ahmad, Ilge Akkaya, Flo-\nrencia Leoni Aleman, Diogo Almeida, Janko Altenschmidt, Sam Altman, Shyamal\nAnadkat, et al. GPT-4 Technical Report.arXiv preprint arXiv:2303.08774, 2023.\n2. Harrison Chase. LangChain, Oct 2022. Available at https://github.com/\nlangchain-ai/langchain.\n3. LangChain, Inc. LangChain Documentation: Integration Providers. LangChain,\nInc.,SanFrancisco,CA,2024. Availableat https://python.langchain.com/docs/\nintegrations/providers/.\n4. LangChain, Inc. LangChain Documentation: Key Concepts. LangChain, Inc.,\nSan Francisco, CA, 2024. Available at https://python.langchain.com/docs/\nconcepts/.\n5. LangChain, Inc. LangChain Documentation: LangServe. LangChain, Inc.,\nSan Francisco, CA, 2024. Available at https://python.langchain.com/docs/\nlangserve/.\n6. LangChain, Inc. LangChain Documentation: Security Best Practices. LangChain,',
  'met

In [54]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage

print("API Key Loaded Successfully")

class GeminiLLM:
    def __init__(self, model_name: str = "gemini-2.5-flash", api_key: str = None):
        

        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GOOGLE_API_KEY")

        if not self.api_key:
            raise ValueError(
                "Google API key is required. Set GOOGLE_API_KEY environment variable."
            )

        self.llm = ChatGoogleGenerativeAI(
            model=self.model_name,
            google_api_key=self.api_key,
            temperature=0.1,
        )

        print(f"Initialized Gemini LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str) -> str:

        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""
You are a Student-friendly AI assistant.

Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""
        )

        formatted_prompt = prompt_template.format(
            context=context,
            question=query
        )

        try:
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content

        except Exception as e:
            return f"Error generating response: {str(e)}"


# Initialize Gemini LLM
try:
    gemini_llm = GeminiLLM(
        api_key=os.getenv("GOOGLE_API_KEY")
    )

    print("Gemini LLM initialized successfully!")

except ValueError as e:
    print(f"Warning: {e}")
    gemini_llm = None

API Key Loaded Successfully
Initialized Gemini LLM with model: gemini-2.5-flash
Gemini LLM initialized successfully!


In [55]:
retrieved_docs = rag_retriever.retrieve(
    "What is LangChain?"
)

Retrieving documents for query: 'What is LangChain?'
Top K: 5, Score threshold: 0.0
generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 103.26it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


In [56]:
print(retrieved_docs)

[{'id': 'doc_3ba6337e_45', 'content': '14 Vasilios Mavroudis\nReferences\n1. Josh Achiam, Steven Adler, Sandhini Agarwal, Lama Ahmad, Ilge Akkaya, Flo-\nrencia Leoni Aleman, Diogo Almeida, Janko Altenschmidt, Sam Altman, Shyamal\nAnadkat, et al. GPT-4 Technical Report.arXiv preprint arXiv:2303.08774, 2023.\n2. Harrison Chase. LangChain, Oct 2022. Available at https://github.com/\nlangchain-ai/langchain.\n3. LangChain, Inc. LangChain Documentation: Integration Providers. LangChain,\nInc.,SanFrancisco,CA,2024. Availableat https://python.langchain.com/docs/\nintegrations/providers/.\n4. LangChain, Inc. LangChain Documentation: Key Concepts. LangChain, Inc.,\nSan Francisco, CA, 2024. Available at https://python.langchain.com/docs/\nconcepts/.\n5. LangChain, Inc. LangChain Documentation: LangServe. LangChain, Inc.,\nSan Francisco, CA, 2024. Available at https://python.langchain.com/docs/\nlangserve/.\n6. LangChain, Inc. LangChain Documentation: Security Best Practices. LangChain,', 'metadat

In [57]:
print(type(retrieved_docs[0]))

<class 'dict'>


In [58]:
context = "\n\n".join(
    [doc["content"] for doc in retrieved_docs]
)

In [59]:
response = gemini_llm.generate_response(
    query="What is LangChain?",
    context=context
)

print(response)

Based on the context, LangChain is a framework or tool that helps developers and researchers build applications using Large Language Models (LLMs) and advance Natural Language Processing (NLP) applications. It has a modular design, extensive integrations, and provides resources like documentation for key concepts, integration providers, LangServe, and security best practices.


In [61]:
response = gemini_llm.generate_response(
    query="What is Retrieval?",
    context=context
)

print(response)

Based on the context, **retrieval** is the process of finding and getting "relevant document chunks" or "contextual information."

This retrieved information is then given to a Large Language Model (LLM) along with the user's question. This way, the LLM can use this external knowledge to generate a response, rather than just relying on what it already knows from its training. This is a key part of what makes RAG (Retrieval Augmented Generation) systems very effective!


In [62]:
response = gemini_llm.generate_response(
    query="What is Augmentation?",
    context=context
)

print(response)

The provided text doesn't directly define "Augmentation," but it explains a process called RAG (Retrieval-Augmented Generation) which is all about augmenting a model's knowledge.

In a RAG system, "augmentation" refers to the process where a Large Language Model (LLM) is given **additional, relevant external information** (like document chunks) along with the user's original question. This means the model doesn't just answer from its own memory, but its knowledge is *enhanced* or *supplemented* by this retrieved context to generate a more informed response.
